In [1]:
import torch
from torch import nn, optim
import pandas as pd

from transformer.data_funcs import calculate_max_mz
from transformer.transformer_utils import load_tokenized_data_with_smiles
from transformer.vocabs import get_or_create_smiles_vocabs
from transformer.sequence_pred import train_model_seq2seq
from transformer.models import MS_VIT_Seq2Seq
from transformer.evaluation import evaluate_model_seq2seq, plot_training_history

In [2]:
if torch.cuda.is_available():
    print("CUDA is available.")
    print("PyTorch version:", torch.__version__)
    print("CUDA version:", torch.version.cuda)
    print("Number of available GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available.")

CUDA is available.
PyTorch version: 2.1.0+cu121
CUDA version: 12.1
Number of available GPUs: 1
GPU name: NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
# df10p = pd.read_csv('data/msms_positive_10ev.csv').drop(columns=['Unnamed: 0'])
# df20p = pd.read_csv('data/msms_positive_10ev.csv').drop(columns=['Unnamed: 0'])
# df40p = pd.read_csv('data/msms_positive_10ev.csv').drop(columns=['Unnamed: 0'])
# df10n = pd.read_csv('data/msms_negative_10ev.csv').drop(columns=['Unnamed: 0'])
# df20n = pd.read_csv('data/msms_negative_10ev.csv').drop(columns=['Unnamed: 0'])
# df40n = pd.read_csv('data/msms_negative_10ev.csv').drop(columns=['Unnamed: 0'])
ir = pd.read_feather('data/ir_spectra_v2.feather')#.drop(columns=['Unnamed: 0'])

In [5]:
ir

,smiles,ir_spectra
0,COc1nc2ccccc2cc1C(=O)O,"[-0.004481, 0.007279, -0.006439, 0.007464, -0...."
1,CCOC(=O)c1cc2c(OCc3coc4cc(F)ccc34)cccc2n1C(=O)...,"[0.028195, 0.011433, 0.024865, -0.00816, 0.055..."
2,CCCCOc1c(CN2C(=O)c3ccccc3C2=O)n(CC2CC2)c(=O)c2...,"[-0.003043, 0.014693, -0.003067, 0.025262, -0...."
3,CCCCNC[C@@H]1O[C@](O)(CO)[C@@H](O)[C@@H]1O,"[0.099321, 0.155452, 0.048156, 0.19118, 0.0237..."
4,O=C(NCC1CC1)c1nc2c(N3CCC(n4c(=O)[nH]c5ccccc54)...,"[0.2335, 0.090925, 0.061724, 0.158822, 0.04248..."
...,...,...
794398,COC(=O)COC(C#C[C@@]1(OC)CN2CCC1CC2)(c1cccs1)c1...,"[0.060733, 0.086449, 0.032809, 0.044489, 0.050..."
794399,CCCCOc1ccc2c(c1-c1ncnc3c(C(=O)NC4CCNCC4)c[nH]c...,"[-0.002789, 0.00637, 0.000991, 0.004822, -0.00..."
794400,N[C@H]1CCCC[C@@H]1OCc1ccccc1,"[0.011316, 0.006252, 0.006748, 0.00923, 0.0157..."
794401,CC(C)(C)c1cc(Cl)n2ncnc2n1,"[0.001994, 0.009854, 0.007437, 0.048094, 0.038..."


In [6]:
msms = pd.read_feather('data/msms_cfmid_positive_40ev_v2.feather')

In [7]:
msms

,smiles,msms_cfmid_positive_40ev
0,COc1nc2ccccc2cc1C(=O)O,"[[101.03858, 100.0], [103.05423, 43.72], [104...."
1,CCOC(=O)c1cc2c(OCc3coc4cc(F)ccc34)cccc2n1C(=O)...,"[[57.06988, 51.14], [91.05423, 9.59], [121.044..."
2,CCCCOc1c(CN2C(=O)c3ccccc3C2=O)n(CC2CC2)c(=O)c2...,"[[57.06988, 28.12], [105.03349, 100.0], [117.0..."
3,CCCCNC[C@@H]1O[C@](O)(CO)[C@@H](O)[C@@H]1O,"[[30.03383, 36.99], [41.03858, 8.43], [42.0338..."
4,O=C(NCC1CC1)c1nc2c(N3CCC(n4c(=O)[nH]c5ccccc54)...,"[[55.05423, 63.08], [70.02874, 18.67], [70.065..."
...,...,...
794398,COC(=O)COC(C#C[C@@]1(OC)CN2CCC1CC2)(c1cccs1)c1...,"[[43.01784, 13.74], [45.03349, 16.66], [57.033..."
794399,CCCCOc1ccc2c(c1-c1ncnc3c(C(=O)NC4CCNCC4)c[nH]c...,"[[57.06988, 32.22], [84.08078, 100.0], [86.096..."
794400,N[C@H]1CCCC[C@@H]1OCc1ccccc1,"[[55.05423, 11.04], [65.03858, 19.43], [79.054..."
794401,CC(C)(C)c1cc(Cl)n2ncnc2n1,"[[57.06988, 62.34], [72.98395, 87.9], [82.0651..."


In [8]:
# Minimal test
n = 50000
df_ir = ir.sample(n)
df_ir_test = ir.drop(index=df_ir.index).sample(n//2)
df_ms = msms.sample(n)
df_ms_test = msms.drop(index=df_ms.index).sample(n//2)

In [9]:
method='direct'

In [11]:
smiles_vocabs = get_or_create_smiles_vocabs(pd.concat([df_ir, df_ir_test, df_ms, df_ms_test]), smiles_col='smiles', source='mfd')

Creating new character vocabulary...
SMILES vocabulary size (character): 38
Creating new atom_wise vocabulary...
SMILES vocabulary size (atom_wise): 14
Creating new substructure vocabulary...
SMILES vocabulary size (substructure): 1838196


In [12]:
max_mz = calculate_max_mz(pd.concat([df_ms, df_ms_test]), spectrum_column='msms_cfmid_positive_40ev')

ValueError: malformed node or string: array([array([56.04948, 60.4    ]), array([70.06513, 23.88   ]),
       array([86.06004, 80.93   ]), array([96.04439, 24.63   ]),
       array([98.06004, 22.75   ]), array([100.07569,  66.26   ]),
       array([105.98543,  31.05   ]), array([108.08078,  41.12   ]),
       array([114.09134,  21.99   ]), array([126.09134,  71.35   ]),
       array([135.05529,  32.53   ]), array([137.07094,  32.29   ]),
       array([139.08659,  39.99   ]), array([144.99633,  28.54   ]),
       array([149.07094,  21.9    ]), array([151.04328,  23.37   ]),
       array([151.08659,  40.87   ]), array([153.05893,  40.73   ]),
       array([168.11314, 100.     ]), array([190.09749,  38.87   ]),
       array([202.05418,  33.22   ]), array([226.13502,  22.51   ]),
       array([229.06508,  32.44   ]), array([243.08073,  67.35   ]),
       array([244.06474,  34.29   ]), array([246.08039,  30.67   ]),
       array([251.13027,  30.89   ]), array([255.08073,  45.76   ]),
      

In [ ]:
results = {}
print(f"\nSpectra tokenized with {method} tokenization:")
print(f"\nSMILES tokenized with {'character'} tokenization")
smiles_vocab = smiles_vocabs['character']

train_loader, test_loader = load_tokenized_data_with_smiles(df, df_test, 
                                                            method, 
                                                            smiles_vocab, 
                                                            max_mz=max_mz)

#num_classes = len(label_encoder.classes_)
smiles_vocab_size = len(smiles_vocab)

# sample batch used for input dimensions
sample_batch, target_batch = next(iter(train_loader))
print("Spectra shape:", sample_batch.shape)
print("SMILES shape:", target_batch.shape)
embed_depth = sample_batch.shape[3]

In [25]:
model = MS_VIT_Seq2Seq(
    smiles_vocab_size=len(smiles_vocab),
    embed_depth=embed_depth,
    d_model=64,           # Reduced from 256
    nhead=4,              # Reduced from 8
    num_layers=2,         # Reduced from 6
    dim_feedforward=256,  # Reduced from 2048
    dropout=0.1,
    num_classes=None
)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion_seq = nn.CrossEntropyLoss()#ignore_index=smiles_vocab['<pad>'])

model, history = train_model_seq2seq(model, train_loader, test_loader, 
                                     optimizer, criterion_seq, 
                                     num_epochs=100, evaluate=True, verbose=1,
                                     checkpoint_path="./model_checkpoints/",
                                     meta_tag=(method+"_character"),
                                     use_tensorboard=True)

In [ ]:
results = evaluate_model_seq2seq(model, test_loader, smiles_vocab, test=True)

In [ ]:
plot_training_history(history)